# Data and Vocabulary Preparation

In [1]:
# Download and decompress text8
# !wget http://mattmahoney.net/dc/text8.zip -O text8.gz
# !gzip -d text8.gz -f

In [2]:
from collections import Counter

# Load and preprocess the corpus
with open("text8", "r") as f:
    text = f.read().lower()
    tokens = text.split()
    
min_freq = 5
word_freq = Counter(tokens) # gets the freq of all unique words
vocab = [(w, c) for w, c in word_freq.items() if c >= min_freq]

In [3]:
vocab.sort(key=lambda x : (-x[1], x[0]))

In [4]:
# Add <UNK> to the vocab. Its frequency is the sum of all rare words.
unk_count = sum(c for w, c in word_freq.items() if c < min_freq)
vocab.append(('<UNK>', unk_count))

In [5]:
word_to_idx = {w: i for i, (w,_) in enumerate(vocab)}
idx_to_word = {i: w for i, (w,_) in enumerate(vocab)}

In [6]:
print(f"Original token count: {len(tokens):,}")
print(f"Filtered vocabulary size: {len(vocab):,}")

Original token count: 17,005,207
Filtered vocabulary size: 71,291


In [10]:
unk_idx = word_to_idx['<UNK>']
full_token_ids = []

for token in tokens:
    # Get the ID. If it's not in our filtered vocab, use <UNK>
    token_id = word_to_idx.get(token, unk_idx)
    full_token_ids.append(token_id)

print(f"Full corpus ID list created. Length: {len(full_token_ids):,}")

Full corpus ID list created. Length: 17,005,207


# SkipGram Training Pair Generation

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
import random

class SkipGramDataset(Dataset):
    """
    A PyTorch Dataset for generating skip-gram (center_word, context_word) pairs.
    """
    def __init__(self, token_ids, window_size):
        """
        Args:
            token_ids (list[int]): The list of token IDs.
            window_size (int): The maximum window size.
        """
        self.token_ids = token_ids
        self.window_size = window_size
        
        # This will store all our (center, context) training pairs
        self.pairs = []
        
        print("Generating training pairs...")
        self._generate_pairs()
        print(f"Generated {len(self.pairs):,} training pairs.")
        
    def _generate_pairs(self):
        """
        Generates all (center, context) pairs and stores them in self.pairs.
        """
        num_tokens = len(self.token_ids)
        
        # Iterate over each token in the corpus to be a center word
        for center_word_idx in range(num_tokens):
            center_word_id = self.token_ids[center_word_idx]
            # Pick a random window size
            current_window_size = random.randint(1, self.window_size)
            
            start_idx = max(0, center_word_idx - current_window_size)
            end_idx   = min(num_tokens, center_word_idx + current_window_size + 1)
            
            for context_word_idx in range(start_idx, end_idx):
                if center_word_idx == context_word_idx:
                    continue
                context_word_id = self.token_ids[context_word_idx]
                self.pairs.append((center_word_id, context_word_id))
    
    def __len__(self):
        """Returns the total number of training pairs."""
        return len(self.pairs)
    
    def __getitem__(self, idx):
        """
        Returns a single (center_word, context_word) pair using tensor.
        """
        center_id, context_id = self.pairs[idx]
        
        # Return as tensors
        return (
            torch.tensor(center_id, dtype=torch.long), 
            torch.tensor(context_id, dtype=torch.long)
        )

In [18]:
WINDOW_SIZE = 5
BATCH_SIZE = 1024

test_token_ids = full_token_ids[:100000]
skipgram_dataset = SkipGramDataset(test_token_ids, window_size=WINDOW_SIZE)

dataloader = DataLoader(
    skipgram_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=2 
)

print("\n--- Testing the DataLoader ---")
center_batch, context_batch = next(iter(dataloader))

print(f"Center batch shape:  {center_batch.shape}")
print(f"Context batch shape: {context_batch.shape}")

print("\nExample center IDs:")
print(center_batch[:5])

print("\nExample context IDs:")
print(context_batch[:5])

Generating training pairs...
Generated 600,356 training pairs.

--- Testing the DataLoader ---
Center batch shape:  torch.Size([1024])
Context batch shape: torch.Size([1024])

Example center IDs:
tensor([15254,   967,  1008, 46940,   180])

Example context IDs:
tensor([ 120,  128,  723, 7443,   16])
